In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)
    
# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

## Env

In [ ]:
from src.utils.env import prepare_environment

prepare_environment("../api_keys")

## Model

In [ ]:
import src.configs.models.art as art_configs
import src.configs.models.vllm as vllm_configs
from src.configs import PathConfig

print("Available ART model configurations:")
print(art_configs.available_configs())

print("Available vLLM model configurations:")
print(vllm_configs.available_configs())

base_model = "unsloth/Qwen3-14B"
project_name = "easy2hard_dac"

path_config = PathConfig(
    base_model=base_model,
    project_name=project_name,
)

In [ ]:
# import art

# host = "0.0.0.0"
# port = 8200

# model = art.Model(
#     name=base_model,
#     project=path_config.project_name,
#     inference_api_key=os.getenv("OPENAI_API_KEY", "default"),
#     inference_base_url=f"https://{host}:{port}/v1",
# )

In [ ]:
from src.utils.loaders import load_art_model

model = await load_art_model(path_config=path_config)

## Data

In [ ]:
from datasets import Dataset, load_dataset, DatasetDict

dataset_dict: DatasetDict = load_dataset(
    path="furonghuang-lab/Easy2Hard-Bench",
    name="E2H-AMC",
    split=None,
)  # type: ignore

train_data: Dataset = dataset_dict["train"]
test_data: Dataset = dataset_dict["eval"]

## Inference Clients

In [ ]:
from src.vllm_client import VllmClient, VllmRouter, ArtClient

inference_clients: list[VllmClient] = [ArtClient.from_art_model(model)]
vllm_server_ports = [8200]
for port in vllm_server_ports:
    inference_clients.append(VllmClient.from_connection(port=port, base_model=base_model))

vllm_router: VllmRouter = VllmRouter(inference_clients)

## Config 

In [ ]:
from experiments.easy2hard_tools.trainer import Easy2HardToolTrainer
from src.configs import PromptConfig, DecompConfig

prompt_config = PromptConfig(
    system_root="dac_sys_prompt_gilad_v2_root_tools",
    system_inter="dac_sys_prompt_gilad_v2_inter_tools",
    system_leaf="dac_sys_prompt_gilad_v2_leaf_tools",
)

decomp_config = DecompConfig(max_depth=1)

trainer = Easy2HardToolTrainer(
    model=model,
    vllm_router=vllm_router,
    path_config=path_config,
    prompt_config=prompt_config,
    decomp_config=decomp_config,
)

### Inference Test

In [ ]:
import random

await trainer.sync_lora()

for i in range(5):
    
    idx = random.randint(0, len(train_data) - 1)
    # idx = 228
    print(f"Selected random index: {idx}")
    sample = train_data[idx]
    problem = sample["problem"]
    true_answer = sample["answer"]

    print(f"Problem: {problem}")
    print(f"Answer: {true_answer}")
    print("-" * 200)
    print()

    preds = await trainer.predict([sample], verbose=True, seed=0, extra_body={"chat_template_kwargs": {"enable_thinking": False}}, stream=False)

In [ ]:
from src.configs import TrainingConfig
from src.utils.io import load_base_model
 
train_config = load_base_model(TrainingConfig, path="../experiments/easy2hard_tools/defaults/train_config.json", do_raise=True)

if "Qwen3" in base_model:
    # Disable thinking for Qwen3 models
    extra_body = train_config.rollout_kwargs["extra_body"] = {"chat_template_kwargs": {"enable_thinking": False}}

await trainer.train(
    train_config,
    train_dataset=train_data.to_list(),
    val_dataset=test_data.to_list(),
)